Homework 10   
Author: Huiping Zhou   
Date: 4/19/2026

# Part 1 - Creating Streaming Data Using rate

First, I will start a new Spark session and set up a streaming DataFrame using the rate format. This provides simple, generated data that can be used to test functionality and explore streaming operations.

In [23]:
#create a spark section
from pyspark.sql import SparkSession         
spark = SparkSession.builder.getOrCreate()

# Sets up a streaming DataFrame using the rate source
rateDF = spark.readStream.format("rate").load()

In [24]:
rateDF

DataFrame[timestamp: timestamp, value: bigint]

We can see the rateDF DataFrame contains two columns: timestamp and value, where value is generated sequentially over time.

Before starting the streaming query, we apply a sequence of transformations using functions from `pyspark.sql.functions` to the rate data. Specifically, these transformations 
- calculate the square root of the `value`    
- find mod 4 of the rate `value`  

These transformations are added to the DataFrame using withColumn, resulting in a new streaming DataFrame that includes both the original columns and the newly computed values.

In [25]:
from pyspark.sql.functions import col, sqrt, pmod

#apply transformation before starting the stream
manipDF = rateDF.withColumn("squaredroot_value", sqrt(col("value"))) \
                            .withColumn("mod4_value", pmod(col("value"), 4))

Next, the transformed streaming DataFrame is written to an output sink. Initially, the stream is written to the `console` in `append` mode to verify that the transformations are being applied correctly. The streaming query is started and then stopped after execution.

In [26]:
debugQuery = (
    manipDF
    .writeStream
    .outputMode("append")
    .format("console")
    .start()
)

#run for 5 seconds
debugQuery.awaitTermination(5)
debugQuery.stop()

26/04/20 17:58:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c9ab788e-cd2f-40cc-8f5e-727b8660eff4. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 17:58:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+-----------------+----------+
|timestamp|value|squaredroot_value|mod4_value|
+---------+-----+-----------------+----------+
+---------+-----+-----------------+----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-----+-----------------+----------+
|           timestamp|value|squaredroot_value|mod4_value|
+--------------------+-----+-----------------+----------+
|2026-04-20 17:58:...|    0|              0.0|         0|
+--------------------+-----+-----------------+----------+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-----+-----------------+----------+
|           timestamp|value|squaredroot_value|mod4_value|
+--------------------+-----+-----------------+----------+
|2026-04-20 17:58:...|    1|              1.0|         

26/04/20 17:58:28 WARN DAGScheduler: Failed to cancel job group f5e11648-7af3-4f7a-9468-ae685b2acf5e. Cannot find active jobs for it.
26/04/20 17:58:28 WARN DAGScheduler: Failed to cancel job group f5e11648-7af3-4f7a-9468-ae685b2acf5e. Cannot find active jobs for it.


Finally, the transformed streaming data is written to an in-memory table by configuring the stream with the `memory` format and assigning a query name `my_table`. This allows the streaming results to be queried using Spark SQL after the stream has run. 

In [27]:
#write stream to memory sink
writeTable = manipDF.writeStream \
                    .format("memory") \
                    .queryName("my_table") \
                    .start()

26/04/20 17:58:34 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bb65a855-c012-40aa-8cef-ac24492a700b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 17:58:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Let it run for about 30 seconds and then stop the query. 

In [28]:
# Let the stream run for ~30 seconds
import time
time.sleep(30)

In [29]:
#stop the stream
writeTable.stop()

26/04/20 17:59:34 WARN DAGScheduler: Failed to cancel job group 5eaec16f-95d0-4273-b558-5b7a779e454c. Cannot find active jobs for it.
26/04/20 17:59:34 WARN DAGScheduler: Failed to cancel job group 5eaec16f-95d0-4273-b558-5b7a779e454c. Cannot find active jobs for it.


Now there is a table that exists for us called my_table. We can use spark.sql() query to access it and display the full contents of the in-memory table .

In [30]:
#output entire in-memory table
spark.sql("select * from my_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value| squaredroot_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-20 17:58:...|    0|               0.0|         0|
|2026-04-20 17:58:...|    1|               1.0|         1|
|2026-04-20 17:58:...|    2|1.4142135623730951|         2|
|2026-04-20 17:58:...|    3|1.7320508075688772|         3|
|2026-04-20 17:58:...|    4|               2.0|         0|
|2026-04-20 17:58:...|    5|  2.23606797749979|         1|
|2026-04-20 17:58:...|    6| 2.449489742783178|         2|
|2026-04-20 17:58:...|    7|2.6457513110645907|         3|
|2026-04-20 17:58:...|    8|2.8284271247461903|         0|
|2026-04-20 17:58:...|    9|               3.0|         1|
|2026-04-20 17:58:...|   10|3.1622776601683795|         2|
|2026-04-20 17:58:...|   11|   3.3166247903554|         3|
|2026-04-20 17:58:...|   12|3.4641016151377544|         0|
|2026-04-20 17:58:...|   13| 3.605551275463989|         

# Part 2 - Using data from a CSV with a Pipeline

In [31]:
#create a spark section
from pyspark.sql import SparkSession         
spark = SparkSession.builder.getOrCreate()

## Read in data
We will use six bikeDetails sub datasets available on the assignment link. We will read in `bikeDetails_for_fit.csv` as a spark (SQL) data frame named `bike` first. 

In [32]:
#read in data as a spark SQL df
bike = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)
bike.show()

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
|    Honda CB Twister|        18000|2010| Individual|1st owner|    60000|            53857|
|Honda CB Hornet 160R|        78500|2018| Individual|1st owner|    17000|            87719|
|Royal Enfield Bul...|       180000|2008| Individual|2nd owner|    39000|       

## Set up Spark ML pipelines 
Using the Spark SQL DataFrame `bike`, we apply an `SQLTransformer` to perform feature transformations. Specifically, we compute the logarithmic transformation of the `selling_price` column and store it as `label`, apply a logarithmic transformation to the `km_driven` column and rename it as `log_km_driven`, and create a binary dummy variable based on the `owner` column. This dummy variable assigns a value of 1 to records corresponding to a first owner and 0 to all other ownership categories, and is stored in the column `one_owner`.

In [33]:
from pyspark.ml.feature import SQLTransformer, VectorAssembler
#Define an SQLTransformer to generate transformed features for modeling
sqltrans = SQLTransformer(
    statement="""
    SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
    FROM __THIS__
    """
    )

Inspect the records of sqltrans

In [34]:
transformed_df = sqltrans.transform(bike)
transformed_df.show(5)

+------------------+----+------------------+---------+
|             label|year|     log_km_driven|one_owner|
+------------------+----+------------------+---------+
|12.072541252905651|2019| 5.857933154483459|        1|
|10.714417768752456|2017| 8.639410824140487|        1|
|11.918390573078392|2018| 9.392661928770137|        1|
|11.082142548877775|2015|10.043249494911286|        1|
| 9.903487552536127|2011|  9.95227771670556|        0|
+------------------+----+------------------+---------+
only showing top 5 rows


Next, a VectorAssembler is used to combine the year, log_km_driven, and one_owner variables into a single `features` column.

In [35]:
#Use a VectorAssembler to create a features column
assembler = VectorAssembler(inputCols = ["year", "log_km_driven", "one_owner"], 
                            outputCol = "features", 
                            handleInvalid = 'keep')

Then, a Pipeline is constructed to chain together the SQLTransformer for feature transformation, the VectorAssembler for feature vector assembly.

In [36]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages = [sqltrans, assembler])

The pipeline is then fitted on the bike DataFrame

In [37]:
#Fit the pipeline and save the object
pipe_fit= pipeline.fit(bike)

The fitted pipeline provides a streamlined and consistent method for preparing the data for future analysis or modeling.

## Data streaming

We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files (the five bikeDetails_add*.csv files).

In [43]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

- The schema from the bike DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [44]:
#set up schema
bike_schema = bike.schema

#set up stream df
stream_df = (
    spark.readStream
         .schema(bike_schema)
         .option("header", True)
         .csv("cvs_bike")
)

- Set up transformation: The previously fitted pipeline is used to transform the incoming streaming data, ensuring the same feature engineering steps are applied as in the static dataset.

In [45]:
#Apply the fitted feature-engineering pipeline to streaming data
stream_transformed = pipe_fit.transform(stream_df)

- The `.writeStream` method is used to output the transformed streaming data to the `console` in `append` mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data.   
- The five CSV files are added to the empty `csv_bike` folder one at a time to be processed incrementally.

In [46]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    stream_transformed
    .writeStream
    .outputMode("append")
    .format("console")
    .start()
)

26/04/20 19:01:37 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-75297cb6-bafa-40c2-9e7a-08d39c3beb6e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 19:01:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

Stop the query after all files are processed

In [47]:
# Stop the streaming query after processing all input files
query.stop()

26/04/20 19:03:19 WARN DAGScheduler: Failed to cancel job group 2d3bee2d-a7bd-41c8-ac25-bd583218b008. Cannot find active jobs for it.
26/04/20 19:03:19 WARN DAGScheduler: Failed to cancel job group 2d3bee2d-a7bd-41c8-ac25-bd583218b008. Cannot find active jobs for it.


The output appears below the streaming query, with five batches corresponding to the five datasets. Each batch includes the engineered columns `label, year, log_km_driven, one_owner, and features`. This workflow prepares the streaming data for future modeling and prediction using a fitted machine learning model.